In [1]:
# starting with content based filtering

In [1]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# loading the data from a CSV file to pandas data
df1 = pd.read_csv("../dataset/tmdb_5000_credits.csv")
df1.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [3]:
# loading the data from a CSV file to pandas data
df2 = pd.read_csv("../dataset/tmdb_5000_movies.csv")
df2.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [84]:
df = df1.merge(df2 , on="title" , how='inner')
df.head(2)

,movie_id,title,cast,crew,budget,genres,homepage,id,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500


In [85]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              4809 non-null   int64  
 1   title                 4809 non-null   object 
 2   cast                  4809 non-null   object 
 3   crew                  4809 non-null   object 
 4   budget                4809 non-null   int64  
 5   genres                4809 non-null   object 
 6   homepage              1713 non-null   object 
 7   id                    4809 non-null   int64  
 8   keywords              4809 non-null   object 
 9   original_language     4809 non-null   object 
 10  original_title        4809 non-null   object 
 11  overview              4806 non-null   object 
 12  popularity            4809 non-null   float64
 13  production_companies  4809 non-null   object 
 14  production_countries  4809 non-null   object 
 15  release_date         

In [86]:
selected_features = ['movie_id' , 'title','overview' , 'genres','keywords','cast','crew']
print(selected_features)
df[selected_features].isnull().sum()

['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']


movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [87]:
# Checking for Null values and filling it with empty string.
# for feature in selected_features:
#     df[feature] = df[feature].fillna('')
df.dropna(inplace=True)
df[selected_features].isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [88]:
df.iloc[0][selected_features]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres      [{"id": 28, "name": "Action"}, {"id": 12, "nam...
keywords    [{"id": 1463, "name": "culture clash"}, {"id":...
cast        [{"cast_id": 242, "character": "Jake Sully", "...
crew        [{"credit_id": "52fe48009251416c750aca23", "de...
Name: 0, dtype: object

In [89]:
import json
def preprocess(x):
    L = []
    for i in x:
        L.append(i["name"])
    return L

def preprocess_cast(x):
    L = []
    for i in x[:3]:
        L.append(i["name"])
    return L
# preprocess(json.loads(df.iloc[0]['genres']))
df['genres'] = df['genres'].apply(json.loads).apply(preprocess)
df['keywords'] = df['keywords'].apply(json.loads).apply(preprocess)
df['cast'] = df['cast'].apply(json.loads).apply(preprocess_cast)
df['crew'] = df['crew'].apply(json.loads).apply(lambda x: [i['name'] for i in x if i['job'] == 'Director'])

In [90]:
df[selected_features].iloc[0]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres          [Action, Adventure, Fantasy, Science Fiction]
keywords    [culture clash, future, space war, space colon...
cast         [Sam Worthington, Zoe Saldana, Sigourney Weaver]
crew                                          [James Cameron]
Name: 0, dtype: object

In [92]:
df['genres'] = df['genres'].apply(lambda x: [i.replace(" ","") for i in x])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
df['cast'] = df['cast'].apply(lambda x: [i.replace(" ","") for i in x])
df['crew'] = df['crew'].apply(lambda x: [i.replace(" ","")  for i in x])

In [93]:
df[selected_features].iloc[0]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres           [Action, Adventure, Fantasy, ScienceFiction]
keywords    [cultureclash, future, spacewar, spacecolony, ...
cast            [SamWorthington, ZoeSaldana, SigourneyWeaver]
crew                                           [JamesCameron]
Name: 0, dtype: object

In [94]:
# Combining all the 5 selected features
combined_string = df['overview'] + " " + df['genres'].apply(lambda x: " ".join(x)) + " " + df['keywords'].apply(lambda x: " ".join(x)) + " " + df['cast'].apply(lambda x: " ".join(x)) + " " + df['crew'].apply(lambda x: " ".join(x))
df['tags'] = combined_string.apply(lambda x: x.lower())

In [95]:
df_new = df[['movie_id','title','tags']]
df_new.head(1)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


In [96]:
# Converting all the Text Data to Numbering vectors
vectorizer = TfidfVectorizer()
feature_vectors = vectorizer.fit_transform(df['tags'])

In [97]:
# Cosine similarity of all the feature vectors
# The Thing it does is that bruteforce From Row1 to all the Rows in the DataFrame.
similarity = cosine_similarity(feature_vectors)

In [98]:
df_new["title"].iloc[1]

"Pirates of the Caribbean: At World's End"

In [99]:
df_new["title"].iloc[1]

"Pirates of the Caribbean: At World's End"

In [100]:
movie_names = df_new['title'].tolist()
movie_names

['Avatar',
 "Pirates of the Caribbean: At World's End",
 'Spectre',
 'The Dark Knight Rises',
 'John Carter',
 'Spider-Man 3',
 'Tangled',
 'Avengers: Age of Ultron',
 'Harry Potter and the Half-Blood Prince',
 'Batman v Superman: Dawn of Justice',
 'Quantum of Solace',
 "Pirates of the Caribbean: Dead Man's Chest",
 'The Lone Ranger',
 'Man of Steel',
 'The Avengers',
 'Pirates of the Caribbean: On Stranger Tides',
 'Men in Black 3',
 'The Hobbit: The Battle of the Five Armies',
 'The Amazing Spider-Man',
 'Robin Hood',
 'The Hobbit: The Desolation of Smaug',
 'The Golden Compass',
 'Titanic',
 'Captain America: Civil War',
 'Jurassic World',
 'Skyfall',
 'Spider-Man 2',
 'Iron Man 3',
 'Alice in Wonderland',
 'Transformers: Revenge of the Fallen',
 'Transformers: Age of Extinction',
 'Oz: The Great and Powerful',
 'The Amazing Spider-Man 2',
 'TRON: Legacy',
 'Cars 2',
 'Green Lantern',
 'Toy Story 3',
 'Terminator Salvation',
 'Furious 7',
 'World War Z',
 'X-Men: Days of Future Pas

In [114]:
# rapidfuzz we are using to find the closest Match.
# Let us say if there are any spelling mistakes in the input array
# we need the correct spelling to find the index and check the cosine similarity
from rapidfuzz import process

movies = df_new['title'].tolist()

user_input = "pirate"

find_close_match = process.extract(user_input, movies , limit=5)

print(find_close_match)

[("Pirates of the Caribbean: At World's End", 75.00000000000001, 1), ("Pirates of the Caribbean: Dead Man's Chest", 75.00000000000001, 11), ('Pirates of the Caribbean: On Stranger Tides', 75.00000000000001, 15), ('The Conspirator', 75.00000000000001, 798), ('Spirited Away', 75.00000000000001, 916)]


In [113]:
df_new.head(1)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


In [122]:
# Our Agenda is to first find the index of the Movie with title.
index_of_the_movie = df_new[df_new.title == list(find_close_match[0])[0]].iloc[0].movie_id
index_of_the_movie

np.int64(285)

In [126]:
# In the similarity_score array we have bruteforce array with all combinations
# of solutions we get list of all movies from the index.
similarity_score = list(enumerate(similarity[index_of_the_movie]))

In [127]:
# So in the below code we are basically sorting in descending order
# to find the Highest Recommended movies
sorted_similar_score = sorted(similarity_score , key= lambda x : x[1] , reverse=True)
sorted_similar_score

[(285, np.float64(1.0000000000000004)),
 (208, np.float64(0.1948923866575835)),
 (87, np.float64(0.13082068982512904)),
 (301, np.float64(0.11560606559076797)),
 (733, np.float64(0.11086255981865376)),
 (16, np.float64(0.1105950589028568)),
 (693, np.float64(0.11047976719546614)),
 (885, np.float64(0.10782882073754609)),
 (160, np.float64(0.104718541667668)),
 (40, np.float64(0.09826014315034357)),
 (1323, np.float64(0.09503131927774697)),
 (643, np.float64(0.0913774547799248)),
 (1017, np.float64(0.08801550660411041)),
 (678, np.float64(0.08549874128720383)),
 (431, np.float64(0.08397177397894162)),
 (366, np.float64(0.08313668459105518)),
 (468, np.float64(0.08307911166028191)),
 (1104, np.float64(0.08190849613022742)),
 (58, np.float64(0.08167719893175104)),
 (101, np.float64(0.0796191030203015)),
 (188, np.float64(0.07765323850165598)),
 (560, np.float64(0.07634802076031483)),
 (125, np.float64(0.07509377630156179)),
 (402, np.float64(0.07470191245204569)),
 (370, np.float64(0.0741

In [128]:
# The below is code for top 20 Highest recommended movies
for i in sorted_similar_score[:40]:
    print(df[df.index == i[0]]['title'])

285    The Other Guys
Name: title, dtype: object
Series([], Name: title, dtype: object)
87    Tomorrowland
Name: title, dtype: object
301    Cloud Atlas
Name: title, dtype: object
Series([], Name: title, dtype: object)
16    The Avengers
Name: title, dtype: object
693    Gone Girl
Name: title, dtype: object
885    The Break-Up
Name: title, dtype: object
160    How to Train Your Dragon 2
Name: title, dtype: object
40    Cars 2
Name: title, dtype: object
1323    City of Ember
Name: title, dtype: object
643    Space Cowboys
Name: title, dtype: object
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
101    X-Men: First Class
Name: title, dtype: object
188    Salt
Name: title, dtype: object
Series([], Name: title, dtype: object)
Series([], Name: title, dtype: object)
402